# Phase 6: Explainable AI (Grad-CAM++)

This phase visualizes **which regions of each dermoscopy image drive the model's predictions**, using a custom Grad-CAM++ implementation built on PyTorch forward/backward hooks (`captum` and `lime` are not installed in this environment, so Grad-CAM++ is implemented directly).

Goals:
- Confirm the model attends to the **lesion itself** rather than borders, hair, rulers, or skin artifacts.
- Compare attention patterns for **correctly classified vs. misclassified malignant cases** (melanoma, basal cell carcinoma) to qualitatively explain the Phase 5 sensitivity numbers.
- Produce figures suitable for the final report (Phase 8).

**Inputs**: `models/checkpoints/final_model.pth`, `results/metrics/test_predictions.csv`, `data/preprocessing_config.json`
**Outputs**: `results/figures/gradcam_examples.png`

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision.models import efficientnet_b4

from config_loader import load_config, resolve_image_path

SEED = 42
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BASE_DIR    = Path(r'c:\graduation project')
MODELS_DIR  = BASE_DIR / 'models' / 'checkpoints'
METRICS_DIR = BASE_DIR / 'results' / 'metrics'
FIGURES_DIR = BASE_DIR / 'results' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

cfg = load_config(BASE_DIR)
CLASSES     = cfg['classes']
NUM_CLASSES = cfg['num_classes']
NORM_MEAN   = cfg['norm_mean']
NORM_STD    = cfg['norm_std']
IMG_SIZE    = cfg['img_size']
MALIGNANT_CLASSES = ['melanoma', 'basal cell carcinoma', 'squamous cell carcinoma']

print(f"Device: {DEVICE}")


class SEBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


class SkinCancerModel(nn.Module):
    IN_FEATURES = 1792

    def __init__(self, num_classes: int = 8):
        super().__init__()
        base = efficientnet_b4(weights=None)
        self.features = base.features
        self.se       = SEBlock(self.IN_FEATURES, reduction=16)
        self.avgpool  = base.avgpool
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(self.IN_FEATURES),
            nn.Linear(self.IN_FEATURES, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.se(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


model = SkinCancerModel(num_classes=NUM_CLASSES).to(DEVICE)
state_dict = torch.load(MODELS_DIR / 'final_model.pth', map_location=DEVICE, weights_only=True)
model.load_state_dict(state_dict)
model.eval()
print("Final model loaded \u2713")

## 1 · Grad-CAM++ Implementation

**Grad-CAM++** (Chattopadhyay et al., 2018) extends Grad-CAM by weighting each spatial location's gradient with a pixel-wise importance term, which gives sharper, more complete localization for lesions that are small or irregularly shaped -- a good fit for dermoscopy images.

**Target layer**: `model.features[-1]`, the final convolutional block of the EfficientNet-B4 backbone. For a 224x224 input this produces a 1792-channel, 7x7 feature map -- the same feature map the SE-attention block re-weights before global pooling and classification.

In [ ]:
class GradCAMPlusPlus:
    """Grad-CAM++ via forward/backward hooks on a target conv layer."""

    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, input_tensor: torch.Tensor, class_idx: int, percentile_clip: float = 90):
        self.model.zero_grad()
        output = self.model(input_tensor)
        score = output[:, class_idx]
        score.backward(retain_graph=True)

        grads = self.gradients
        acts  = self.activations

        grads2 = grads ** 2
        grads3 = grads2 * grads
        sum_acts = acts.sum(dim=(2, 3), keepdim=True)
        eps = 1e-8
        alpha = grads2 / (2 * grads2 + sum_acts * grads3 + eps)
        alpha = torch.where(grads != 0, alpha, torch.zeros_like(alpha))
        weights = (alpha * F.relu(grads)).sum(dim=(2, 3), keepdim=True)

        cam = (weights * acts).sum(dim=1, keepdim=True)
        cam = F.relu(cam).squeeze().cpu().numpy()

        # EfficientNet feature maps often contain a single outlier cell (e.g. a
        # corner) with a magnitude far above the rest of the grid. Left alone,
        # min-max normalization collapses everything else to near-zero. Clip to
        # a percentile of the low-resolution CAM before upsampling so the
        # lesion-relevant activations remain visible.
        cap = np.percentile(cam, percentile_clip)
        if cap > 0:
            cam = np.minimum(cam, cap)

        cam_t = torch.from_numpy(cam).float().unsqueeze(0).unsqueeze(0)
        cam_t = F.interpolate(cam_t, size=input_tensor.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam_t.squeeze().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + eps)
        return cam, output.softmax(dim=1).detach().cpu().numpy()[0]


gradcam = GradCAMPlusPlus(model, model.features[-1])
print("Grad-CAM++ ready \u2713")

## 2 · Visualization Helpers

Loads a preprocessed image with the same resize/normalize transform used for inference (no augmentation), and overlays the Grad-CAM++ heatmap using OpenCV's JET colormap.

In [ ]:
inference_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=NORM_MEAN, std=NORM_STD),
])


def load_image_tensor(path: str):
    image = Image.open(path).convert('RGB')
    image = image.resize((IMG_SIZE, IMG_SIZE))
    tensor = inference_transform(image).unsqueeze(0).to(DEVICE)
    return image, tensor


def overlay_heatmap(image_pil: Image.Image, cam: np.ndarray, alpha: float = 0.45):
    image_np = np.asarray(image_pil).astype(np.float32) / 255.0
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    overlay = alpha * heatmap + (1 - alpha) * image_np
    return np.clip(overlay, 0, 1)


print("Helpers defined \u2713")

## 3 · Select Example Cases

Using `test_predictions.csv` from Phase 5, pick representative cases:
- **Melanoma**: 2 confidently-correct (TP) + 2 confidently-misclassified (FN) -- the FN cases are the ones driving the gap between argmax and threshold-adjusted melanoma sensitivity.
- **Basal cell carcinoma**: 1 TP + 1 misclassified, for the same reason.
- **Nevus**: 1 confidently-correct case as a benign reference.

In [ ]:
pred_df = pd.read_csv(METRICS_DIR / 'test_predictions.csv')

proba_cols = [f"proba_{c.replace(' ', '_')}" for c in CLASSES]
pred_df['pred_proba'] = pred_df[proba_cols].max(axis=1)


def pick(label: str, correct: bool, n: int):
    if correct:
        subset = pred_df[(pred_df['label_str'] == label) & (pred_df['pred_str'] == label)]
        sort_col = f"proba_{label.replace(' ', '_')}"
    else:
        subset = pred_df[(pred_df['label_str'] == label) & (pred_df['pred_str'] != label)]
        sort_col = 'pred_proba'
    return subset.sort_values(sort_col, ascending=False).head(n)


examples = pd.concat([
    pick('melanoma', correct=True, n=2),
    pick('melanoma', correct=False, n=2),
    pick('basal cell carcinoma', correct=True, n=1),
    pick('basal cell carcinoma', correct=False, n=1),
    pick('nevus', correct=True, n=1),
]).reset_index(drop=True)

examples[['image_id', 'label_str', 'pred_str', 'pred_proba']]

## 4 · Grad-CAM++ Visualizations

For each example, the heatmap explains the model's **predicted** class (so for misclassified melanomas, it shows what made the model think "nevus" or another class).

In [ ]:
n = len(examples)
fig, axes = plt.subplots(n, 3, figsize=(9, 3.0 * n))

col_titles = ['Original', 'Grad-CAM++', 'Overlay']

for row_idx, (_, row) in enumerate(examples.iterrows()):
    path = resolve_image_path(row, cfg)
    image_pil, tensor = load_image_tensor(path)

    pred_idx = CLASSES.index(row['pred_str'])
    cam, proba = gradcam.generate(tensor, pred_idx)
    overlay = overlay_heatmap(image_pil, cam)

    correct = row['label_str'] == row['pred_str']
    tag = 'correct' if correct else 'MISCLASSIFIED'
    label_text = (
        f"True: {row['label_str']}\n"
        f"Pred: {row['pred_str']} ({proba[pred_idx]:.2f})\n"
        f"[{tag}]"
    )

    axes[row_idx, 0].imshow(image_pil)
    axes[row_idx, 0].axis('off')
    axes[row_idx, 0].text(
        -0.05, 0.5, label_text, transform=axes[row_idx, 0].transAxes,
        fontsize=9, va='center', ha='right',
    )

    axes[row_idx, 1].imshow(cam, cmap='jet')
    axes[row_idx, 1].axis('off')

    axes[row_idx, 2].imshow(overlay)
    axes[row_idx, 2].axis('off')

for j, title in enumerate(col_titles):
    axes[0, j].set_title(title, fontsize=11, fontweight='bold')

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'gradcam_examples.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {FIGURES_DIR / 'gradcam_examples.png'}")

## Phase 6 Complete ✓

**Output**: `results/figures/gradcam_examples.png` -- Grad-CAM++ heatmaps for melanoma (TP/FN), basal cell carcinoma (TP/FN), and a benign nevus reference.

**How to read the figure**:
- If the heatmap concentrates on the **lesion** for true-positive cases, the model has learned clinically plausible features.
- For **false-negative melanomas**, check whether the heatmap is diffuse, focused on healthy skin, or drawn to artifacts (hair, rulers, vignetting) -- this can explain why those cases were missed and motivates the threshold-adjusted decision rule from Phase 5.

**Optional follow-up**: a LIME superpixel explanation could complement Grad-CAM++ but requires `pip install lime` (not currently installed) -- deferred unless needed for the report.

**Next**: Phase 7 (RAG-based report assistant).